In [1]:
# Updated feature extraction aligned with RanSMAP paper
import os
import pandas as pd
import numpy as np
from glob import glob

WINDOW_SIZE = 10  # seconds
STEP_SIZE = 10    # seconds
ROOT = "/home/henrykabuye/Ransap/Datasets/RanSMAP/dataset/original/i3-gen12/ddr4-2133-8g"
BATCHED_OUTPUT_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched"

CSV_STRUCTURE = {
    "ata_read.csv":       ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "ata_write.csv":      ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "mem_exec.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_read.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_readwrite.csv":  ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_write.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"]
}

# Feature extraction helpers
def safe_var(series):
    return series.var() if not series.empty else 0

def extract_ransmap_features(df, ftype):
    features = {}
    # Throughput
    throughput = df['size'].sum() / WINDOW_SIZE
    if 'ata_read' in ftype:
        features['avg_read_throughput'] = throughput
        features['read_LBA_variance'] = safe_var(df['LBA'])
    elif 'ata_write' in ftype:
        features['avg_write_throughput'] = throughput
        features['write_LBA_variance'] = safe_var(df['LBA'])
        features['write_entropy_mean'] = df['entropy'].mean()
    elif 'mem_' in ftype:
        op = ftype.split('.')[0].split('_')[1]  # read, write, exec, readwrite
        op_prefix = f"mem_{op}"
        features[f"{op_prefix}_GPA_variance"] = safe_var(df['GPA'])
        if op in ['write', 'readwrite']:
            features[f"{op_prefix}_entropy_mean"] = df['size'].apply(lambda x: -(x * np.log2(x + 1e-10))).mean()
        features[f"{op_prefix}_count"] = len(df)
        if 'MMIO_flag' in df.columns:
            features['mem_exec_count'] = df['MMIO_flag'].sum()
        # Access types
        if 'access_type' in df.columns:
            for atype in [2, 4]:
                features[f"access_type_{atype}_count"] = (df['access_type'] == atype).sum()
        if 'page_type' in df.columns:
            features['mem_4KiB_page_access_count'] = (df['page_type'] == -0.0).sum()
    return features

def extract_features_from_folder(folder_path, class_label, instance, save_path):
    all_features = []
    for ftype, columns in CSV_STRUCTURE.items():
        fpath = os.path.join(folder_path, ftype)
        if not os.path.exists(fpath):
            continue
        df = pd.read_csv(fpath, names=columns, header=0)
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])
        t_min, t_max = df['timestamp'].min(), df['timestamp'].max()
        for start in np.arange(t_min, t_max, STEP_SIZE):
            end = start + WINDOW_SIZE
            window_df = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]
            if window_df.empty:
                continue
            feats = extract_ransmap_features(window_df, ftype)
            feats['file_type'] = ftype.replace('.csv', '')
            feats['bucket_start_time'] = start
            feats['label'] = class_label
            all_features.append(feats)
    if all_features:
        pd.DataFrame(all_features).to_parquet(save_path, index=False)
        print(f"✅ Saved: {save_path}")

def process_all_data(root=ROOT):
    for label_folder in os.listdir(root):
        label_path = os.path.join(root, label_folder)
        if not os.path.isdir(label_path):
            continue
        for instance in os.listdir(label_path):
            instance_path = os.path.join(label_path, instance)
            if not os.path.isdir(instance_path):
                continue
            save_path = os.path.join(BATCHED_OUTPUT_DIR, f"features_{label_folder}_{instance}.parquet")
            extract_features_from_folder(instance_path, label_folder, instance, save_path)

def combine_all_batches(directory):
    def encode_binary_label(df):
        benign = ['Idle', 'Firefox', 'Office', 'SDelete', 'AESCrypt', 'Zip']
        df['label'] = df['label'].apply(lambda x: 0 if x in benign else 1)
        return df
    files = glob(os.path.join(directory, '*.parquet'))
    df_list = [encode_binary_label(pd.read_parquet(f)) for f in files]
    combined = pd.concat(df_list, ignore_index=True)
    nulls = combined.isnull().mean()
    drop_cols = nulls[nulls > 0.8].index.tolist()
    combined.drop(columns=drop_cols, inplace=True)
    for col in combined.select_dtypes(include='number'):
        combined[col].fillna(combined[col].median(), inplace=True)
    combined.to_parquet(os.path.join(directory, "combined_i3-gen12_dataset.parquet"))
    print("✅ Combined dataset saved.")

# Run
process_all_data()
combine_all_batches(BATCHED_OUTPUT_DIR)


✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-37-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-53-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-20-03.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-56-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-11-58.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_21-00-09.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-46-34.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-32-31.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-20-52.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-03-19.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/featur

/tmp/ipykernel_275885/496783756.py:101: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined[col].fillna(combined[col].median(), inplace=True)


In [5]:
import pandas as pd

# Load the Parquet file (requires pyarrow or fastparquet)
df = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_dataset.parquet")  # adjust path if needed

# Save as CSV
df.to_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_dataset.csv", index=False)

print("Conversion complete. Saved as 'combined_i3-gen12_dataset.csv'")


Conversion complete. Saved as 'combined_i3-gen12_dataset.csv'


In [6]:
# Updated feature extraction aligned with RanSMAP paper (Full Coverage)
import os
import pandas as pd
import numpy as np
from glob import glob

WINDOW_SIZE = 10  # seconds
STEP_SIZE = 10    # seconds
ROOT = "/home/henrykabuye/Ransap/Datasets/RanSMAP/dataset/original/i3-gen12/ddr4-2133-8g"
BATCHED_OUTPUT_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched"

CSV_STRUCTURE = {
    "ata_read.csv":       ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "ata_write.csv":      ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "mem_exec.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_read.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_readwrite.csv":  ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_write.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"]
}

# Feature extraction helpers
def safe_var(series):
    return series.var() if not series.empty else 0

def shannon_entropy(series):
    return -(series * np.log2(series + 1e-10)).mean() if not series.empty else 0

def extract_ransmap_features(df, ftype):
    features = {}
    throughput = df['size'].sum() / WINDOW_SIZE
    if 'ata_read' in ftype:
        features['avg_read_throughput'] = throughput
        features['read_LBA_variance'] = safe_var(df['LBA'])
    elif 'ata_write' in ftype:
        features['avg_write_throughput'] = throughput
        features['write_LBA_variance'] = safe_var(df['LBA'])
        features['write_entropy_mean'] = df['entropy'].mean()
    elif 'mem_' in ftype:
        op = ftype.split('.')[0].split('_')[1]  # read, write, exec, readwrite
        op_prefix = f"mem_{op}"
        features[f"{op_prefix}_GPA_variance"] = safe_var(df['GPA'])
        features[f"{op_prefix}_count"] = len(df)
        if op in ['read', 'write', 'readwrite']:
            features[f"{op_prefix}_entropy_mean"] = shannon_entropy(df['size'])
        if op == 'exec' and 'MMIO_flag' in df.columns:
            features['mem_exec_count'] = df['MMIO_flag'].sum()
        if 'access_type' in df.columns:
            for atype in [2, 4]:
                features[f"access_type_{atype}_count"] = (df['access_type'] == atype).sum()
        if 'page_type' in df.columns:
            features['mem_4KiB_page_access_count'] = (df['page_type'] == -0.0).sum()
    return features

def extract_features_from_folder(folder_path, class_label, instance, save_path):
    all_features = []
    for ftype, columns in CSV_STRUCTURE.items():
        fpath = os.path.join(folder_path, ftype)
        if not os.path.exists(fpath):
            continue
        df = pd.read_csv(fpath, names=columns, header=0)
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])
        t_min, t_max = df['timestamp'].min(), df['timestamp'].max()
        for start in np.arange(t_min, t_max, STEP_SIZE):
            end = start + WINDOW_SIZE
            window_df = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]
            if window_df.empty:
                continue
            feats = extract_ransmap_features(window_df, ftype)
            feats['file_type'] = ftype.replace('.csv', '')
            feats['bucket_start_time'] = start
            feats['label'] = class_label
            all_features.append(feats)
    if all_features:
        pd.DataFrame(all_features).to_parquet(save_path, index=False)
        print(f"✅ Saved: {save_path}")

def process_all_data(root=ROOT):
    for label_folder in os.listdir(root):
        label_path = os.path.join(root, label_folder)
        if not os.path.isdir(label_path):
            continue
        for instance in os.listdir(label_path):
            instance_path = os.path.join(label_path, instance)
            if not os.path.isdir(instance_path):
                continue
            save_path = os.path.join(BATCHED_OUTPUT_DIR, f"features_{label_folder}_{instance}.parquet")
            extract_features_from_folder(instance_path, label_folder, instance, save_path)

def combine_all_batches(directory):
    def encode_binary_label(df):
        benign = ['Idle', 'Firefox', 'Office', 'SDelete', 'AESCrypt', 'Zip']
        df['label'] = df['label'].apply(lambda x: 0 if x in benign else 1)
        return df
    files = glob(os.path.join(directory, '*.parquet'))
    df_list = [encode_binary_label(pd.read_parquet(f)) for f in files]
    combined = pd.concat(df_list, ignore_index=True)
    nulls = combined.isnull().mean()
    drop_cols = nulls[nulls > 0.8].index.tolist()
    combined.drop(columns=drop_cols, inplace=True)
    for col in combined.select_dtypes(include='number'):
        combined[col].fillna(combined[col].median(), inplace=True)
    combined.to_parquet(os.path.join(directory, "combined_i3-gen12_datasetB.parquet"))
    print("✅ Combined dataset saved.")

# Run
process_all_data()
combine_all_batches(BATCHED_OUTPUT_DIR)


✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-37-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-53-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-20-03.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-56-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-11-58.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_21-00-09.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-46-34.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-32-31.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-20-52.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-03-19.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/featur

/tmp/ipykernel_275885/182363450.py:102: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined[col].fillna(combined[col].median(), inplace=True)


In [7]:
import pandas as pd

# Load the Parquet file (requires pyarrow or fastparquet)
df = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetB.parquet")  # adjust path if needed

# Save as CSV
df.to_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetB.csv", index=False)

print("Conversion complete. Saved as 'combined_i3-gen12_datasetB.csv'")


Conversion complete. Saved as 'combined_i3-gen12_datasetB.csv'


In [8]:
# Updated feature extraction aligned with RanSMAP paper (Complete + Debug Logging)
import os
import pandas as pd
import numpy as np
from glob import glob

WINDOW_SIZE = 10  # seconds
STEP_SIZE = 10    # seconds
ROOT = "/home/henrykabuye/Ransap/Datasets/RanSMAP/dataset/original/i3-gen12/ddr4-2133-8g"
BATCHED_OUTPUT_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched"

CSV_STRUCTURE = {
    "ata_read.csv":       ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "ata_write.csv":      ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "mem_exec.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_read.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_readwrite.csv":  ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_write.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"]
}

# Feature extraction helpers
def safe_var(series):
    return series.var() if not series.empty else 0

def shannon_entropy(series):
    return -(series * np.log2(series + 1e-10)).mean() if not series.empty else 0

def extract_ransmap_features(df, ftype):
    features = {}
    throughput = df['size'].sum() / WINDOW_SIZE
    if 'ata_read' in ftype:
        features['avg_read_throughput'] = throughput
        features['read_LBA_variance'] = safe_var(df['LBA'])
    elif 'ata_write' in ftype:
        features['avg_write_throughput'] = throughput
        features['write_LBA_variance'] = safe_var(df['LBA'])
        features['write_entropy_mean'] = df['entropy'].mean()
    elif 'mem_' in ftype:
        op = ftype.split('.')[0].split('_')[1]  # read, write, exec, readwrite
        op_prefix = f"mem_{op}"
        features[f"{op_prefix}_GPA_variance"] = safe_var(df['GPA'])
        features[f"{op_prefix}_count"] = len(df)
        if op in ['read', 'write', 'readwrite']:
            features[f"{op_prefix}_entropy_mean"] = shannon_entropy(df['size'])
        if op == 'exec' and 'MMIO_flag' in df.columns:
            features['mem_exec_count'] = df['MMIO_flag'].sum()
        if 'access_type' in df.columns:
            for atype in [2, 4]:
                features[f"access_type_{atype}_count"] = (df['access_type'] == atype).sum()
        if 'page_type' in df.columns:
            features['mem_4KiB_page_access_count'] = (df['page_type'] == -0.0).sum()
    print(f"🔍 Extracted from {ftype}: {list(features.keys())}")
    return features

def extract_features_from_folder(folder_path, class_label, instance, save_path):
    all_features = []
    for ftype, columns in CSV_STRUCTURE.items():
        fpath = os.path.join(folder_path, ftype)
        if not os.path.exists(fpath):
            print(f"⚠ Skipping missing file: {fpath}")
            continue
        df = pd.read_csv(fpath, names=columns, header=0)
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])
        t_min, t_max = df['timestamp'].min(), df['timestamp'].max()
        for start in np.arange(t_min, t_max, STEP_SIZE):
            end = start + WINDOW_SIZE
            window_df = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]
            if window_df.empty:
                continue
            feats = extract_ransmap_features(window_df, ftype)
            feats['file_type'] = ftype.replace('.csv', '')
            feats['bucket_start_time'] = start
            feats['label'] = class_label
            all_features.append(feats)
    if all_features:
        pd.DataFrame(all_features).to_parquet(save_path, index=False)
        print(f"✅ Saved: {save_path}")

def process_all_data(root=ROOT):
    for label_folder in os.listdir(root):
        label_path = os.path.join(root, label_folder)
        if not os.path.isdir(label_path):
            continue
        for instance in os.listdir(label_path):
            instance_path = os.path.join(label_path, instance)
            if not os.path.isdir(instance_path):
                continue
            save_path = os.path.join(BATCHED_OUTPUT_DIR, f"features_{label_folder}_{instance}.parquet")
            extract_features_from_folder(instance_path, label_folder, instance, save_path)

def combine_all_batches(directory):
    def encode_binary_label(df):
        benign = ['Idle', 'Firefox', 'Office', 'SDelete', 'AESCrypt', 'Zip']
        df['label'] = df['label'].apply(lambda x: 0 if x in benign else 1)
        return df
    files = glob(os.path.join(directory, '*.parquet'))
    df_list = [encode_binary_label(pd.read_parquet(f)) for f in files]
    combined = pd.concat(df_list, ignore_index=True)
    nulls = combined.isnull().mean()
    drop_cols = nulls[nulls > 0.8].index.tolist()
    combined.drop(columns=drop_cols, inplace=True)
    for col in combined.select_dtypes(include='number'):
        combined[col].fillna(combined[col].median(), inplace=True)
    combined.to_parquet(os.path.join(directory, "combined_i3-gen12_datasetC.parquet"))
    print("✅ Combined dataset saved.")

# Run
process_all_data()
combine_all_batches(BATCHED_OUTPUT_DIR)


🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted 

/tmp/ipykernel_275885/2444862370.py:104: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined[col].fillna(combined[col].median(), inplace=True)


In [9]:
import pandas as pd

# Load the Parquet file (requires pyarrow or fastparquet)
df = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetC.parquet")  # adjust path if needed

# Save as CSV
df.to_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetC.csv", index=False)

print("Conversion complete. Saved as 'combined_i3-gen12_datasetC.csv'")


Conversion complete. Saved as 'combined_i3-gen12_datasetC.csv'


In [11]:
# Updated feature extraction aligned with RanSMAP paper (Complete + Debug Logging + Column Consistency)
import os
import pandas as pd
import numpy as np
from glob import glob

WINDOW_SIZE = 10  # seconds
STEP_SIZE = 10    # seconds
ROOT = "/home/henrykabuye/Ransap/Datasets/RanSMAP/dataset/original/i3-gen12/ddr4-2133-8g"
BATCHED_OUTPUT_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched"

CSV_STRUCTURE = {
    "ata_read.csv":       ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "ata_write.csv":      ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "mem_exec.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_read.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_readwrite.csv":  ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_write.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"]
}

ALL_FEATURE_KEYS = [
    'avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance', 'write_entropy_mean',
    'mem_read_entropy_mean', 'mem_write_entropy_mean', 'mem_readwrite_entropy_mean',
    'mem_read_GPA_variance', 'mem_write_GPA_variance', 'mem_readwrite_GPA_variance',
    'mem_exec_GPA_variance',
    'mem_read_count', 'mem_write_count', 'mem_readwrite_count', 'mem_exec_count',
    'access_type_2_count', 'access_type_4_count', 'mem_4KiB_page_access_count'
]

# Feature extraction helpers
def safe_var(series):
    return series.var() if not series.empty else 0

def shannon_entropy(series):
    return -(series * np.log2(series + 1e-10)).mean() if not series.empty else 0

def extract_ransmap_features(df, ftype):
    features = {}
    throughput = df['size'].sum() / WINDOW_SIZE
    if 'ata_read' in ftype:
        features['avg_read_throughput'] = throughput
        features['read_LBA_variance'] = safe_var(df['LBA'])
    elif 'ata_write' in ftype:
        features['avg_write_throughput'] = throughput
        features['write_LBA_variance'] = safe_var(df['LBA'])
        features['write_entropy_mean'] = df['entropy'].mean()
    elif 'mem_' in ftype:
        op = ftype.split('.')[0].split('_')[1]  # read, write, exec, readwrite
        op_prefix = f"mem_{op}"
        features[f"{op_prefix}_GPA_variance"] = safe_var(df['GPA'])
        features[f"{op_prefix}_count"] = len(df)
        if op in ['read', 'write', 'readwrite']:
            features[f"{op_prefix}_entropy_mean"] = shannon_entropy(df['size'])
        if op == 'exec' and 'MMIO_flag' in df.columns:
            features['mem_exec_count'] = df['MMIO_flag'].sum()
        if 'access_type' in df.columns:
            for atype in [2, 4]:
                features[f"access_type_{atype}_count"] = (df['access_type'] == atype).sum()
        if 'page_type' in df.columns:
            features['mem_4KiB_page_access_count'] = (df['page_type'] == -0.0).sum()
    print(f"🔍 Extracted from {ftype}: {list(features.keys())}")
    return features

def extract_features_from_folder(folder_path, class_label, instance, save_path):
    all_features = []
    for ftype, columns in CSV_STRUCTURE.items():
        fpath = os.path.join(folder_path, ftype)
        if not os.path.exists(fpath):
            print(f"⚠ Skipping missing file: {fpath}")
            continue
        df = pd.read_csv(fpath, names=columns, header=0)
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])
        t_min, t_max = df['timestamp'].min(), df['timestamp'].max()
        for start in np.arange(t_min, t_max, STEP_SIZE):
            end = start + WINDOW_SIZE
            window_df = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]
            if window_df.empty:
                continue
            feats = extract_ransmap_features(window_df, ftype)
            feats['file_type'] = ftype.replace('.csv', '')
            feats['bucket_start_time'] = start
            feats['label'] = class_label
            # Ensure all expected feature keys are present
            for key in ALL_FEATURE_KEYS:
                feats.setdefault(key, np.nan)
            all_features.append(feats)
    if all_features:
        pd.DataFrame(all_features).to_parquet(save_path, index=False)
        print(f"✅ Saved: {save_path}")

def process_all_data(root=ROOT):
    for label_folder in os.listdir(root):
        label_path = os.path.join(root, label_folder)
        if not os.path.isdir(label_path):
            continue
        for instance in os.listdir(label_path):
            instance_path = os.path.join(label_path, instance)
            if not os.path.isdir(instance_path):
                continue
            save_path = os.path.join(BATCHED_OUTPUT_DIR, f"features_{label_folder}_{instance}.parquet")
            extract_features_from_folder(instance_path, label_folder, instance, save_path)

def combine_all_batches(directory):
    def encode_binary_label(df):
        benign = ['Idle', 'Firefox', 'Office', 'SDelete', 'AESCrypt', 'Zip']
        df['label'] = df['label'].apply(lambda x: 0 if x in benign else 1)
        return df
    files = glob(os.path.join(directory, '*.parquet'))
    df_list = [encode_binary_label(pd.read_parquet(f)) for f in files]
    combined = pd.concat(df_list, ignore_index=True)
    nulls = combined.isnull().mean()
    drop_cols = nulls[nulls > 0.8].index.tolist()
    combined.drop(columns=drop_cols, inplace=True)
    for col in combined.select_dtypes(include='number'):
        combined[col].fillna(combined[col].median(), inplace=True)
    combined.to_parquet(os.path.join(directory, "combined_i3-gen12_datasetD.parquet"))
    print("✅ Combined dataset saved.")

# Run
process_all_data()
combine_all_batches(BATCHED_OUTPUT_DIR)


🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted from ata_read.csv: ['avg_read_throughput', 'read_LBA_variance']
🔍 Extracted 

/tmp/ipykernel_275885/3557493260.py:117: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined[col].fillna(combined[col].median(), inplace=True)


In [12]:
import pandas as pd

# Load the Parquet file (requires pyarrow or fastparquet)
df = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetD.parquet")  # adjust path if needed

# Save as CSV
df.to_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetD.csv", index=False)

print("Conversion complete. Saved as 'combined_i3-gen12_datasetD.csv'")


Conversion complete. Saved as 'combined_i3-gen12_datasetD.csv'


In [13]:
# Finalized RanSMAP Feature Extraction: Unified Window-Based Aggregation
import os
import pandas as pd
import numpy as np
from glob import glob

WINDOW_SIZE = 10  # seconds
STEP_SIZE = 10    # seconds
ROOT = "/home/henrykabuye/Ransap/Datasets/RanSMAP/dataset/original/i3-gen12/ddr4-2133-8g"
BATCHED_OUTPUT_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched"

CSV_STRUCTURE = {
    "ata_read.csv":       ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "ata_write.csv":      ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "mem_exec.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_read.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_readwrite.csv":  ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_write.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"]
}

ALL_FEATURE_KEYS = [
    'avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance', 'write_entropy_mean',
    'mem_read_entropy_mean', 'mem_write_entropy_mean', 'mem_readwrite_entropy_mean',
    'mem_read_GPA_variance', 'mem_write_GPA_variance', 'mem_readwrite_GPA_variance', 'mem_exec_GPA_variance',
    'mem_read_count', 'mem_write_count', 'mem_readwrite_count', 'mem_exec_count',
    'access_type_2_count', 'access_type_4_count', 'mem_4KiB_page_access_count'
]

# Feature extraction helpers
def safe_var(series):
    return series.var() if not series.empty else 0

def shannon_entropy(series):
    return -(series * np.log2(series + 1e-10)).mean() if not series.empty else 0

def extract_features_per_window(windows, ftype, df):
    features_by_window = {}
    for start, end in windows:
        win_df = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]
        feats = {}
        if win_df.empty:
            features_by_window[start] = feats
            continue

        if 'ata_read' in ftype:
            feats['avg_read_throughput'] = win_df['size'].sum() / WINDOW_SIZE
            feats['read_LBA_variance'] = safe_var(win_df['LBA'])
        elif 'ata_write' in ftype:
            feats['avg_write_throughput'] = win_df['size'].sum() / WINDOW_SIZE
            feats['write_LBA_variance'] = safe_var(win_df['LBA'])
            feats['write_entropy_mean'] = win_df['entropy'].mean()
        elif 'mem_' in ftype:
            op = ftype.split('.')[0].split('_')[1]
            op_prefix = f"mem_{op}"
            feats[f"{op_prefix}_GPA_variance"] = safe_var(win_df['GPA'])
            feats[f"{op_prefix}_count"] = len(win_df)
            if op in ['read', 'write', 'readwrite']:
                feats[f"{op_prefix}_entropy_mean"] = shannon_entropy(win_df['size'])
            if op == 'exec':
                feats['mem_exec_GPA_variance'] = safe_var(win_df['GPA'])
                feats['mem_exec_count'] = win_df['MMIO_flag'].sum()
            if 'access_type' in win_df.columns:
                for atype in [2, 4]:
                    feats[f"access_type_{atype}_count"] = (win_df['access_type'] == atype).sum()
            if 'page_type' in win_df.columns:
                feats['mem_4KiB_page_access_count'] = (win_df['page_type'] == -0.0).sum()
        features_by_window[start] = feats
    return features_by_window

def extract_features_from_folder(folder_path, class_label, instance, save_path):
    dfs = {}
    for ftype, cols in CSV_STRUCTURE.items():
        fpath = os.path.join(folder_path, ftype)
        if os.path.exists(fpath):
            df = pd.read_csv(fpath, names=cols, header=0)
            df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
            dfs[ftype] = df.dropna(subset=['timestamp'])
        else:
            print(f"⚠ Missing file: {ftype}")

    all_times = [df['timestamp'] for df in dfs.values() if not df.empty]
    if not all_times:
        print(f"⚠ No data in folder: {folder_path}")
        return

    t_min = min(s.min() for s in all_times)
    t_max = max(s.max() for s in all_times)
    windows = [(start, start + WINDOW_SIZE) for start in np.arange(t_min, t_max, STEP_SIZE)]

    all_window_features = []
    for start, end in windows:
        win_features = {key: np.nan for key in ALL_FEATURE_KEYS}
        for ftype, df in dfs.items():
            if df.empty:
                continue
            extracted = extract_features_per_window([(start, end)], ftype, df)[start]
            win_features.update(extracted)
        win_features['bucket_start_time'] = start
        win_features['label'] = class_label
        all_window_features.append(win_features)

    df_out = pd.DataFrame(all_window_features)
    df_out.to_parquet(save_path, index=False)
    print(f"✅ Saved: {save_path}")

def process_all_data(root=ROOT):
    for label_folder in os.listdir(root):
        label_path = os.path.join(root, label_folder)
        if not os.path.isdir(label_path):
            continue
        for instance in os.listdir(label_path):
            instance_path = os.path.join(label_path, instance)
            if not os.path.isdir(instance_path):
                continue
            save_path = os.path.join(BATCHED_OUTPUT_DIR, f"features_{label_folder}_{instance}.parquet")
            extract_features_from_folder(instance_path, label_folder, instance, save_path)

def combine_all_batches(directory):
    def encode_binary_label(df):
        benign = ['Idle', 'Firefox', 'Office', 'SDelete', 'AESCrypt', 'Zip']
        df['label'] = df['label'].apply(lambda x: 0 if x in benign else 1)
        return df
    files = glob(os.path.join(directory, '*.parquet'))
    df_list = [encode_binary_label(pd.read_parquet(f)) for f in files]
    combined = pd.concat(df_list, ignore_index=True)
    nulls = combined.isnull().mean()
    drop_cols = nulls[nulls > 0.8].index.tolist()
    combined.drop(columns=drop_cols, inplace=True)
    for col in combined.select_dtypes(include='number'):
        combined[col].fillna(combined[col].median(), inplace=True)
    combined.to_parquet(os.path.join(directory, "combined_i3-gen12_datasetE.parquet"))
    print("✅ Combined dataset saved.")

# Run
process_all_data()
combine_all_batches(BATCHED_OUTPUT_DIR)


✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-37-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-53-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-20-03.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-56-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-11-58.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_21-00-09.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-46-34.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-32-31.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-20-52.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-03-19.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/featur

/tmp/ipykernel_275885/3880487334.py:131: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined[col].fillna(combined[col].median(), inplace=True)


In [14]:
import pandas as pd

# Load the Parquet file (requires pyarrow or fastparquet)
df = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetE.parquet")  # adjust path if needed

# Save as CSV
df.to_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetE.csv", index=False)

print("Conversion complete. Saved as 'combined_i3-gen12_datasetE.csv'")


Conversion complete. Saved as 'combined_i3-gen12_datasetE.csv'


In [15]:
# Finalized RanSMAP Feature Extraction: Unified Window-Based Aggregation (Complete and Corrected)
import os
import pandas as pd
import numpy as np
from glob import glob

WINDOW_SIZE = 10  # seconds
STEP_SIZE = 10    # seconds
ROOT = "/home/henrykabuye/Ransap/Datasets/RanSMAP/dataset/original/i3-gen12/ddr4-2133-8g"
BATCHED_OUTPUT_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched"

CSV_STRUCTURE = {
    "ata_read.csv":       ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "ata_write.csv":      ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "mem_exec.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_read.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_readwrite.csv":  ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_write.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"]
}

ALL_FEATURE_KEYS = [
    'avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance', 'write_entropy_mean',
    'mem_read_entropy_mean', 'mem_write_entropy_mean', 'mem_readwrite_entropy_mean',
    'mem_read_GPA_variance', 'mem_write_GPA_variance', 'mem_readwrite_GPA_variance', 'mem_exec_GPA_variance',
    'mem_read_count', 'mem_write_count', 'mem_readwrite_count', 'mem_exec_count',
    'access_type_2_count', 'access_type_4_count', 'mem_4KiB_page_access_count'
]

# Feature extraction helpers
def safe_var(series):
    return series.var() if not series.empty else 0

def shannon_entropy(series):
    return -(series * np.log2(series + 1e-10)).mean() if not series.empty else 0

def extract_features_per_window(windows, ftype, df):
    features_by_window = {}
    for start, end in windows:
        win_df = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]
        feats = {}
        if win_df.empty:
            features_by_window[start] = feats
            continue

        if 'ata_read' in ftype:
            feats['avg_read_throughput'] = win_df['size'].sum() / WINDOW_SIZE
            feats['read_LBA_variance'] = safe_var(win_df['LBA'])
        elif 'ata_write' in ftype:
            feats['avg_write_throughput'] = win_df['size'].sum() / WINDOW_SIZE
            feats['write_LBA_variance'] = safe_var(win_df['LBA'])
            feats['write_entropy_mean'] = win_df['entropy'].mean()
        elif 'mem_' in ftype:
            op = ftype.split('.')[0].split('_')[1]
            op_prefix = f"mem_{op}"
            feats[f"{op_prefix}_GPA_variance"] = safe_var(win_df['GPA'])
            feats[f"{op_prefix}_count"] = len(win_df)
            if op in ['read', 'write', 'readwrite']:
                feats[f"{op_prefix}_entropy_mean"] = shannon_entropy(win_df['size'])
            if op == 'exec' and 'MMIO_flag' in win_df.columns:
                feats['mem_exec_count'] = win_df['MMIO_flag'].sum()
            if 'access_type' in win_df.columns:
                for atype in [2, 4]:
                    key = f"access_type_{atype}_count"
                    feats[key] = (win_df['access_type'] == atype).sum()
            if 'page_type' in win_df.columns:
                feats['mem_4KiB_page_access_count'] = (win_df['page_type'] == -0.0).sum()
        features_by_window[start] = feats
    return features_by_window

def extract_features_from_folder(folder_path, class_label, instance, save_path):
    dfs = {}
    for ftype, cols in CSV_STRUCTURE.items():
        fpath = os.path.join(folder_path, ftype)
        if os.path.exists(fpath):
            df = pd.read_csv(fpath, names=cols, header=0)
            df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
            dfs[ftype] = df.dropna(subset=['timestamp'])
        else:
            print(f"⚠ Missing file: {ftype}")

    all_times = [df['timestamp'] for df in dfs.values() if not df.empty]
    if not all_times:
        print(f"⚠ No data in folder: {folder_path}")
        return

    t_min = min(s.min() for s in all_times)
    t_max = max(s.max() for s in all_times)
    windows = [(start, start + WINDOW_SIZE) for start in np.arange(t_min, t_max, STEP_SIZE)]

    all_window_features = []
    for start, end in windows:
        win_features = {key: np.nan for key in ALL_FEATURE_KEYS}
        for ftype, df in dfs.items():
            if df.empty:
                continue
            extracted = extract_features_per_window([(start, end)], ftype, df)[start]
            win_features.update(extracted)
        win_features['bucket_start_time'] = start
        win_features['label'] = class_label
        all_window_features.append(win_features)

    df_out = pd.DataFrame(all_window_features)
    df_out.to_parquet(save_path, index=False)
    print(f"✅ Saved: {save_path}")

def process_all_data(root=ROOT):
    for label_folder in os.listdir(root):
        label_path = os.path.join(root, label_folder)
        if not os.path.isdir(label_path):
            continue
        for instance in os.listdir(label_path):
            instance_path = os.path.join(label_path, instance)
            if not os.path.isdir(instance_path):
                continue
            save_path = os.path.join(BATCHED_OUTPUT_DIR, f"features_{label_folder}_{instance}.parquet")
            extract_features_from_folder(instance_path, label_folder, instance, save_path)

def combine_all_batches(directory):
    def encode_binary_label(df):
        benign = ['Idle', 'Firefox', 'Office', 'SDelete', 'AESCrypt', 'Zip']
        df['label'] = df['label'].apply(lambda x: 0 if x in benign else 1)
        return df
    files = glob(os.path.join(directory, '*.parquet'))
    df_list = [encode_binary_label(pd.read_parquet(f)) for f in files]
    combined = pd.concat(df_list, ignore_index=True)
    nulls = combined.isnull().mean()
    drop_cols = nulls[nulls > 0.8].index.tolist()
    combined.drop(columns=drop_cols, inplace=True)
    for col in combined.select_dtypes(include='number'):
        combined[col].fillna(combined[col].median(), inplace=True)
    combined.to_parquet(os.path.join(directory, "combined_dataset.parquet"))
    print("✅ Combined dataset saved.")

# Run
process_all_data()
combine_all_batches(BATCHED_OUTPUT_DIR)


✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-37-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-53-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-20-03.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-56-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-11-58.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_21-00-09.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-46-34.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-32-31.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-20-52.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-03-19.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/featur

/tmp/ipykernel_275885/3277793785.py:131: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined[col].fillna(combined[col].median(), inplace=True)


In [17]:
import pandas as pd

# Load the Parquet file (requires pyarrow or fastparquet)
df = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_Conti_Conti-20230705_21-05-28.parquet")  # adjust path if needed

# Save as CSV
df.to_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_Conti_Conti-20230705_21-05-28.csv", index=False)

print("Conversion complete. Saved as 'features_Conti_Conti-20230705_21-05-28.csv'")


Conversion complete. Saved as 'features_Conti_Conti-20230705_21-05-28.csv'


In [18]:
# Finalized RanSMAP Feature Extraction: Unified Window-Based Aggregation (Complete and Corrected)
import os
import pandas as pd
import numpy as np
from glob import glob

WINDOW_SIZE = 10  # seconds
STEP_SIZE = 10    # seconds
ROOT = "/home/henrykabuye/Ransap/Datasets/RanSMAP/dataset/original/i3-gen12/ddr4-2133-8g"
BATCHED_OUTPUT_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched"

CSV_STRUCTURE = {
    "ata_read.csv":       ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "ata_write.csv":      ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "mem_exec.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_read.csv":       ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_readwrite.csv":  ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_write.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"]
}

ALL_FEATURE_KEYS = [
    'avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance', 'write_entropy_mean',
    'mem_read_entropy_mean', 'mem_write_entropy_mean', 'mem_readwrite_entropy_mean',
    'mem_read_GPA_variance', 'mem_write_GPA_variance', 'mem_readwrite_GPA_variance', 'mem_exec_GPA_variance',
    'mem_read_count', 'mem_write_count', 'mem_readwrite_count', 'mem_exec_count',
    'access_type_2_count', 'access_type_4_count', 'mem_4KiB_page_access_count'
]

# Feature extraction helpers
def safe_var(series):
    return series.var() if not series.empty else 0

def shannon_entropy(series):
    return -(series * np.log2(series + 1e-10)).mean() if not series.empty else 0

def extract_features_per_window(windows, ftype, df):
    features_by_window = {}
    for start, end in windows:
        win_df = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]
        feats = {}
        if win_df.empty:
            features_by_window[start] = feats
            continue

        if 'ata_read' in ftype:
            feats['avg_read_throughput'] = win_df['size'].sum() / WINDOW_SIZE
            feats['read_LBA_variance'] = safe_var(win_df['LBA'])
        elif 'ata_write' in ftype:
            feats['avg_write_throughput'] = win_df['size'].sum() / WINDOW_SIZE
            feats['write_LBA_variance'] = safe_var(win_df['LBA'])
            feats['write_entropy_mean'] = win_df['entropy'].mean()
        elif 'mem_' in ftype:
            op = ftype.split('.')[0].split('_')[1]
            op_prefix = f"mem_{op}"
            feats[f"{op_prefix}_GPA_variance"] = safe_var(win_df['GPA'])
            feats[f"{op_prefix}_count"] = len(win_df)
            if op in ['read', 'write', 'readwrite']:
                feats[f"{op_prefix}_entropy_mean"] = shannon_entropy(win_df['size'])
            if op == 'exec' and 'MMIO_flag' in win_df.columns:
                feats['mem_exec_count'] = win_df['MMIO_flag'].sum()
            if 'access_type' in win_df.columns:
                for atype in [2, 4]:
                    key = f"access_type_{atype}_count"
                    feats[key] = (win_df['access_type'] == atype).sum()
            if 'page_type' in win_df.columns:
                feats['mem_4KiB_page_access_count'] = (win_df['page_type'] == -0.0).sum()
        features_by_window[start] = feats
    return features_by_window

def extract_features_from_folder(folder_path, class_label, instance, save_path):
    dfs = {}
    for ftype, cols in CSV_STRUCTURE.items():
        fpath = os.path.join(folder_path, ftype)
        if os.path.exists(fpath):
            df = pd.read_csv(fpath, names=cols, header=0)
            df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
            dfs[ftype] = df.dropna(subset=['timestamp'])
        else:
            print(f"⚠ Missing file: {ftype}")

    all_times = [df['timestamp'] for df in dfs.values() if not df.empty]
    if not all_times:
        print(f"⚠ No data in folder: {folder_path}")
        return

    t_min = min(s.min() for s in all_times)
    t_max = max(s.max() for s in all_times)
    windows = [(start, start + WINDOW_SIZE) for start in np.arange(t_min, t_max, STEP_SIZE)]

    all_window_features = []
    for start, end in windows:
        win_features = {key: np.nan for key in ALL_FEATURE_KEYS}
        for ftype, df in dfs.items():
            if df.empty:
                continue
            extracted = extract_features_per_window([(start, end)], ftype, df)[start]
            win_features.update(extracted)
        win_features['bucket_start_time'] = start
        win_features['label'] = class_label
        all_window_features.append(win_features)

    df_out = pd.DataFrame(all_window_features)
    df_out.to_parquet(save_path, index=False)
    print(f"✅ Saved: {save_path}")

def process_all_data(root=ROOT):
    for label_folder in os.listdir(root):
        label_path = os.path.join(root, label_folder)
        if not os.path.isdir(label_path):
            continue
        for instance in os.listdir(label_path):
            instance_path = os.path.join(label_path, instance)
            if not os.path.isdir(instance_path):
                continue
            save_path = os.path.join(BATCHED_OUTPUT_DIR, f"features_{label_folder}_{instance}.parquet")
            extract_features_from_folder(instance_path, label_folder, instance, save_path)

def combine_all_batches(directory):
    def encode_binary_label(df):
        benign = ['Idle', 'Firefox', 'Office', 'SDelete', 'AESCrypt', 'Zip']
        df['label'] = df['label'].apply(lambda x: 0 if x in benign else 1)
        return df

    files = glob(os.path.join(directory, '*.parquet'))
    df_list = []
    for f in files:
        df = pd.read_parquet(f)
        df = encode_binary_label(df)
        for col in ALL_FEATURE_KEYS + ['bucket_start_time', 'label']:
            if col not in df.columns:
                df[col] = np.nan
        df_list.append(df)

    combined = pd.concat(df_list, ignore_index=True)[ALL_FEATURE_KEYS + ['bucket_start_time', 'label']]

    for col in combined.select_dtypes(include='number'):
        combined[col].fillna(combined[col].median(), inplace=True)

    combined.to_parquet(os.path.join(directory, "combined_i3-gen12_datasetF.parquet"))
    print("✅ Combined dataset saved with enforced columns.")

# Run
process_all_data()
combine_all_batches(BATCHED_OUTPUT_DIR)

✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-37-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-53-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-20-03.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-56-02.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-11-58.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_21-00-09.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-46-34.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-32-31.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230712_20-20-52.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/features_SDelete_SDelete-20230928_23-03-19.parquet
✅ Saved: /mnt/4TBSTORE/Dr_Kabuye4TB/Batched/featur

/tmp/ipykernel_275885/2929554879.py:138: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined[col].fillna(combined[col].median(), inplace=True)


✅ Combined dataset saved with enforced columns.


In [19]:
import pandas as pd

# Load the Parquet file (requires pyarrow or fastparquet)
df = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet")  # adjust path if needed

# Save as CSV
df.to_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.csv", index=False)

print("Conversion complete. Saved as 'combined_i3-gen12_datasetF.csv'")


Conversion complete. Saved as 'combined_i3-gen12_datasetF.csv'


In [20]:
import pandas as pd

df = pd.read_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.csv")
print(df.dtypes)
print(df.head(3))

avg_read_throughput           float64
avg_write_throughput          float64
read_LBA_variance             float64
write_LBA_variance            float64
write_entropy_mean            float64
mem_read_entropy_mean         float64
mem_write_entropy_mean        float64
mem_readwrite_entropy_mean    float64
mem_read_GPA_variance         float64
mem_write_GPA_variance        float64
mem_readwrite_GPA_variance    float64
mem_exec_GPA_variance         float64
mem_read_count                float64
mem_write_count               float64
mem_readwrite_count           float64
mem_exec_count                float64
access_type_2_count           float64
access_type_4_count           float64
mem_4KiB_page_access_count    float64
bucket_start_time               int64
label                           int64
dtype: object
   avg_read_throughput  avg_write_throughput  read_LBA_variance  \
0         1.203471e+11          2.322224e+10       9.175133e+16   
1         1.811373e+11          1.078178e+11       9.5

In [22]:
df = pd.read_csv("/home/henrykabuye/Ransap/Datasets/Ransomware_Detection_ModelA1-20241205T121601Z-001/Ransomware_Detection_ModelA1/Ransomware.csv", delimiter='|')
print(df.dtypes)
print(df.head(3))

Name                            object
md5                             object
Machine                          int64
SizeOfOptionalHeader             int64
Characteristics                  int64
MajorLinkerVersion               int64
MinorLinkerVersion               int64
SizeOfCode                       int64
SizeOfInitializedData            int64
SizeOfUninitializedData          int64
AddressOfEntryPoint              int64
BaseOfCode                       int64
BaseOfData                       int64
ImageBase                      float64
SectionAlignment                 int64
FileAlignment                    int64
MajorOperatingSystemVersion      int64
MinorOperatingSystemVersion      int64
MajorImageVersion                int64
MinorImageVersion                int64
MajorSubsystemVersion            int64
MinorSubsystemVersion            int64
SizeOfImage                      int64
SizeOfHeaders                    int64
CheckSum                         int64
Subsystem                

In [27]:
# Select representative fields
text_columns = ["Name", "ImportsNbDLL", "SectionsMaxEntropy", "ResourcesMinEntropy", "VersionInformationSize"]

# Convert all to strings and join into a semantic sentence
df["semantic_text"] = df[text_columns].astype(str).agg(" ".join, axis=1)

# Example output
print(df["semantic_text"].head())


0      memtest.exe 0 7.22105072892 2.56884382364 16
1          ose.exe 7 6.56690933416 3.42074425303 18
2       setup.exe 14 7.60095678141 2.84644858861 18
3        DW20.EXE 15 7.59492880283 2.66931388802 18
4    dwtrig20.exe 10 7.60830453625 3.42159769887 18
Name: semantic_text, dtype: object


In [1]:
import pandas as pd

from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import torch

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load the CSV with pipe delimiter
df = pd.read_csv("/mnt/4TBSTORE/Dr_Kabuye4TB/RDset/Ransomware.csv", delimiter="|", engine="python")

# Convert label (1 = benign -> 0, 0 = malware -> 1)
df["label"] = df["legitimate"].apply(lambda x: 0 if x == 1 else 1)

# Drop unwanted fields
df = df.drop(columns=["md5", "legitimate", "SizeOfOptionalHeader", "MajorLinkerVersion", "MinorLinkerVersion",
    "BaseOfCode", "BaseOfData", "ImageBase", "SectionAlignment", "FileAlignment",
    "CheckSum", "SizeOfHeapReserve", "SizeOfHeapCommit", "SizeOfStackReserve", "SizeOfStackCommit",
    "LoaderFlags", "NumberOfRvaAndSizes"], errors="ignore")

# Define semantic columns to combine as text
semantic_columns = [
    "Name", "Machine", "Characteristics",
    "ImportsNbDLL", "ImportsNb", "ExportsNb",
    "SectionsMeanEntropy", "SectionsMaxEntropy", "SectionsMinEntropy",
    "ResourcesNb", "ResourcesMeanEntropy", "ResourcesMinEntropy",
    "ResourcesMaxEntropy", "ResourcesMeanSize", "ResourcesMinSize",
    "ResourcesMaxSize", "LoadConfigurationSize", "VersionInformationSize",
    "Subsystem", "DllCharacteristics"
]

# Ensure they exist
semantic_columns = [col for col in semantic_columns if col in df.columns]

# Create a semantic text field
df["semantic_text"] = df[semantic_columns].astype(str).agg(" ".join, axis=1)

# Load SentenceTransformer model
model = SentenceTransformer("paraphrase-MiniLM-L6-v2", device=device)

# Generate embeddings
print("Encoding semantic_text...")
df["embedding"] = model.encode(df["semantic_text"].tolist(), show_progress_bar=True, convert_to_numpy=True).tolist()

# Save to Parquet
output_path = "/home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet"
df.to_parquet(output_path, index=False)
print("✅ Embeddings saved to:", output_path)


/home/henrykabuye/anaconda3/envs/kabuye1_env/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Using device: cuda


/home/henrykabuye/anaconda3/envs/kabuye1_env/lib/python3.9/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Encoding semantic_text...


Batches:   0%|          | 0/4314 [00:00<?, ?it/s]

✅ Embeddings saved to: /home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet


In [13]:
# === Imports ===
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset
import optuna
from imblearn.over_sampling import SMOTE

import random

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)


# === Device Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_tensor(x): return torch.tensor(x, dtype=torch.float32, device=device)

# === Load Semantic Dataset ===
df_sem = pd.read_parquet("/home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet")
X_sem = np.stack(df_sem["embedding"].values).astype(np.float32)
y_sem = df_sem["label"].astype(np.float32).values
print("Semantic Embedding Shape:", X_sem.shape)
print("Semantic Labels Shape:", y_sem.shape)

# === Load Behavioral Dataset ===
df_beh = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet")
behavior_columns = [
    'avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance',
    'write_entropy_mean', 'mem_read_entropy_mean', 'mem_write_entropy_mean',
    'mem_readwrite_entropy_mean', 'mem_read_GPA_variance',
    'mem_write_GPA_variance', 'mem_readwrite_GPA_variance',
    'mem_exec_GPA_variance', 'mem_read_count', 'mem_write_count',
    'mem_readwrite_count', 'mem_exec_count'
]
X_beh = df_beh[behavior_columns].to_numpy(dtype=np.float32)
y_beh = df_beh["label"].astype(np.float32).values
print("Behavioral Features Shape:", X_beh.shape)
print("Behavioral Labels Shape:", y_beh.shape)

# === Define Models ===
class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.model(x).squeeze()

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.model(x).squeeze()

class DecisionAgent(nn.Module):
    def __init__(self, sem_dim, beh_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(sem_dim + beh_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, s, b): return self.model(torch.cat([s, b], dim=1)).squeeze()

# === Train Semantic Agent ===
def train_semantic_agent(X, y, input_dim):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    model = CNNSemanticAgent(input_dim, dropout=0.3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test)), batch_size=256)

    model.train()
    for _ in range(15):
        for xb, yb in train_loader:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())
    print("Semantic Agent Accuracy:", accuracy_score(targets, preds))
    return model



sm = SMOTE(random_state=42)
X_beh_bal, y_beh_bal = sm.fit_resample(X_beh, y_beh)

print("After SMOTE:")
print("Balanced X shape:", X_beh_bal.shape)
print("Balanced y distribution:", np.bincount(y_beh_bal.astype(int)))


# === Train Behavioral Agent ===
def train_behavioral_agent(X, y, input_dim):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    model = CNNBehavioralAgent(input_dim, dropout=0.24323230212272448).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002849614138701997)
    # Calculate class weight dynamically
    pos_weight = torch.tensor([len(y) / sum(y) - 1]).to(device)  # ratio of neg:pos

    # criterion = nn.BCELoss(weight=None, reduction='mean')  # Optional
    # For more stability in logits mode (recommended):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # criterion = nn.BCELoss()

    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test)), batch_size=256)

    model.train()
    for _ in range(15):
        for xb, yb in train_loader:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())
    print("Behavioral Agent Accuracy:", accuracy_score(targets, preds))
    return model

def objective(trial):
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 10, 30)

    model = CNNBehavioralAgent(X_beh.shape[1], dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    X_train, X_test, y_train, y_test = train_test_split(X_beh, y_beh, test_size=0.2, stratify=y_beh, random_state=42)
    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test)), batch_size=256)

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())

    return accuracy_score(targets, preds)

# === Train Both Agents ===
sem_model = train_semantic_agent(X_sem, y_sem, X_sem.shape[1])
beh_model = train_behavioral_agent(X_beh, y_beh, X_beh.shape[1])
beh_model = train_behavioral_agent(X_beh_bal, y_beh_bal, X_beh.shape[1])


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best Hyperparameters:")
print(study.best_trial.params)

print(f"Epoch {_}, Loss: {loss.item():.4f}")





Semantic Embedding Shape: (138047, 384)
Semantic Labels Shape: (138047,)
Behavioral Features Shape: (751588, 16)
Behavioral Labels Shape: (751588,)
After SMOTE:
Balanced X shape: (1500658, 16)
Balanced y distribution: [750329 750329]
Semantic Agent Accuracy: 1.0
Behavioral Agent Accuracy: 0.0016764459346186086


[I 2025-05-22 16:44:35,580] A new study created in memory with name: no-name-3ce4b0f8-be6c-4885-83e3-dbc0ad9e5e2e


Behavioral Agent Accuracy: 0.5


[I 2025-05-22 16:48:06,037] Trial 0 finished with value: 0.03782647454064051 and parameters: {'dropout': 0.3253827764823628, 'lr': 0.0023381648067301117, 'epochs': 22}. Best is trial 0 with value: 0.03782647454064051.
[I 2025-05-22 16:50:19,048] Trial 1 finished with value: 0.998330206628614 and parameters: {'dropout': 0.22086583288008207, 'lr': 0.0009417525999187645, 'epochs': 14}. Best is trial 1 with value: 0.998330206628614.
[I 2025-05-22 16:54:48,704] Trial 2 finished with value: 0.998330206628614 and parameters: {'dropout': 0.12567360402043692, 'lr': 0.0011585549719107077, 'epochs': 28}. Best is trial 1 with value: 0.998330206628614.
[I 2025-05-22 16:57:13,735] Trial 3 finished with value: 0.9982969438124509 and parameters: {'dropout': 0.16405678724183254, 'lr': 0.0011923109581426171, 'epochs': 15}. Best is trial 1 with value: 0.998330206628614.
[I 2025-05-22 16:59:07,260] Trial 4 finished with value: 0.9931611649968733 and parameters: {'dropout': 0.353337857161768, 'lr': 0.00605

Best Hyperparameters:
{'dropout': 0.22086583288008207, 'lr': 0.0009417525999187645, 'epochs': 14}
Epoch 14, Loss: 0.0001


In [12]:
# === Extract features from output of semantic/behavioral models (prediction probabilities) ===
def extract_agent_outputs(model, X, batch_size=256):
    loader = DataLoader(to_tensor(X).unsqueeze(1), batch_size=batch_size)
    outputs = []
    model.eval()
    with torch.no_grad():
        for xb in loader:
            preds = model(xb)
            # Convert prediction to 2D array to keep compatibility
            outputs.append(preds.unsqueeze(1).cpu().numpy())
    return np.vstack(outputs)

# === Use output predictions as input to decision agent ===
semantic_outputs = extract_agent_outputs(sem_model, X_sem)
behavioral_outputs = extract_agent_outputs(beh_model, X_beh)

# === Align lengths (smallest common set) ===
min_len = min(len(semantic_outputs), len(behavioral_outputs))
X_dec = np.concatenate([semantic_outputs[:min_len], behavioral_outputs[:min_len]], axis=1)
y_dec = y_sem[:min_len]  # Labels from semantic set

# === Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(X_dec, y_dec, test_size=0.2, stratify=y_dec, random_state=42)

# === Define Decision Agent ===
class DecisionAgent(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.model(x).squeeze()

# === Train Decision Agent ===
dec_model = DecisionAgent(input_dim=X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(dec_model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

train_loader = DataLoader(TensorDataset(to_tensor(X_train), to_tensor(y_train)), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(to_tensor(X_test), to_tensor(y_test)), batch_size=256)

# === Training Loop ===
for epoch in range(15):
    dec_model.train()
    for xb, yb in train_loader:
        pred = dec_model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

# === Evaluation ===
dec_model.eval()
preds, targets = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        p = dec_model(xb)
        preds.extend((p > 0.5).int().cpu().numpy())
        targets.extend(yb.cpu().numpy())

# === Report Accuracy ===
acc = accuracy_score(targets, preds)
print("Decision Agent Accuracy:", acc)


Decision Agent Accuracy: 1.0


In [10]:
# === Imports ===
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset

# === Device Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_tensor(x): return torch.tensor(x, dtype=torch.float32, device=device)

# === Load Semantic Dataset ===
df_sem = pd.read_parquet("/home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet")
X_sem = np.stack(df_sem["embedding"].values).astype(np.float32)
y_sem = df_sem["label"].astype(np.float32).values
print("Semantic Embedding Shape:", X_sem.shape)
print("Semantic Labels Shape:", y_sem.shape)

# === Load Behavioral Dataset ===
df_beh = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet")
behavior_columns = [
    'avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance',
    'write_entropy_mean', 'mem_read_entropy_mean', 'mem_write_entropy_mean',
    'mem_readwrite_entropy_mean', 'mem_read_GPA_variance',
    'mem_write_GPA_variance', 'mem_readwrite_GPA_variance',
    'mem_exec_GPA_variance', 'mem_read_count', 'mem_write_count',
    'mem_readwrite_count', 'mem_exec_count'
]
X_beh = df_beh[behavior_columns].to_numpy(dtype=np.float32)
y_beh = df_beh["label"].astype(np.float32).values
print("Behavioral Features Shape:", X_beh.shape)
print("Behavioral Labels Shape:", y_beh.shape)

# === Define Models ===
class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature(x)
        return self.classifier(features).squeeze(), features

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature(x)
        return self.classifier(features).squeeze(), features

class DecisionAgent(nn.Module):
    def __init__(self, sem_dim, beh_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(sem_dim + beh_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, s, b): return self.model(torch.cat([s, b], dim=1)).squeeze()

# === Utility Training Function ===
def train_binary_agent(model, X, y, input_dim, extract_features=False):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002849614138701997)
    criterion = nn.BCELoss()

    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test)), batch_size=256)

    model.train()
    for _ in range(15):
        for xb, yb in train_loader:
            pred, _ = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    preds, targets, sem_feats = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p, f = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())
            if extract_features:
                sem_feats.extend(f.cpu().numpy())
    acc = accuracy_score(targets, preds)
    return model, acc, np.array(sem_feats) if extract_features else None

# === Train Agents ===
sem_model = CNNSemanticAgent(X_sem.shape[1], dropout=0.3).to(device)
sem_model, acc_sem, sem_features = train_binary_agent(sem_model, X_sem, y_sem, X_sem.shape[1], extract_features=True)
print("Semantic Agent Accuracy:", acc_sem)

beh_model = CNNBehavioralAgent(X_beh.shape[1], dropout=0.24323230212272448).to(device)
beh_model, acc_beh, beh_features = train_binary_agent(beh_model, X_beh, y_beh, X_beh.shape[1], extract_features=True)
print("Behavioral Agent Accuracy:", acc_beh)

# === Train Decision Agent on matched subset ===
min_len = min(len(sem_features), len(beh_features))
X_dec = np.concatenate([sem_features[:min_len], beh_features[:min_len]], axis=1)
y_dec = y_sem[:min_len]  # Assuming semantic labels are reliable

X_train, X_test, y_train, y_test = train_test_split(X_dec, y_dec, test_size=0.2, stratify=y_dec, random_state=42)
dec_model = DecisionAgent(32, 32, dropout=0.3).to(device)
optimizer = torch.optim.Adam(dec_model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
train_loader = DataLoader(TensorDataset(to_tensor(X_train), to_tensor(y_train)), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(to_tensor(X_test), to_tensor(y_test)), batch_size=256)

# === Train ===
dec_model.train()
for _ in range(15):
    for xb, yb in train_loader:
        pred = dec_model(xb[:, :32], xb[:, 32:])
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

# === Evaluate ===
dec_model.eval()
preds, targets = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        p = dec_model(xb[:, :32], xb[:, 32:])
        preds.extend((p > 0.5).int().cpu().numpy())
        targets.extend(yb.cpu().numpy())

print("Decision Agent Accuracy:", accuracy_score(targets, preds))


Semantic Embedding Shape: (138047, 384)
Semantic Labels Shape: (138047,)
Behavioral Features Shape: (751588, 16)
Behavioral Labels Shape: (751588,)
Semantic Agent Accuracy: 1.0
Behavioral Agent Accuracy: 0.9559932942162616
Decision Agent Accuracy: 0.014125316914161535


In [5]:
# === Imports ===
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset

# === Device Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_tensor(x): return torch.tensor(x, dtype=torch.float32, device=device)

# === Load Semantic Dataset ===
df_sem = pd.read_parquet("/home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet")
X_sem = np.stack(df_sem["embedding"].values).astype(np.float32)
y_sem = df_sem["label"].astype(np.float32).values
print("Semantic Embedding Shape:", X_sem.shape)
print("Semantic Labels Shape:", y_sem.shape)

# === Load Behavioral Dataset ===
df_beh = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet")
behavior_columns = [
    'avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance',
    'write_entropy_mean', 'mem_read_entropy_mean', 'mem_write_entropy_mean',
    'mem_readwrite_entropy_mean', 'mem_read_GPA_variance',
    'mem_write_GPA_variance', 'mem_readwrite_GPA_variance',
    'mem_exec_GPA_variance', 'mem_read_count', 'mem_write_count',
    'mem_readwrite_count', 'mem_exec_count'
]
X_beh = df_beh[behavior_columns].to_numpy(dtype=np.float32)
y_beh = df_beh["label"].astype(np.float32).values
print("Behavioral Features Shape:", X_beh.shape)
print("Behavioral Labels Shape:", y_beh.shape)

# === Define Models ===
class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature(x)
        return self.classifier(features).squeeze(), features

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature(x)
        return self.classifier(features).squeeze(), features

class DecisionAgent(nn.Module):
    def __init__(self, sem_dim, beh_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(sem_dim + beh_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, s, b): return self.model(torch.cat([s, b], dim=1)).squeeze()

# === Utility Training Function ===
def train_binary_agent(model, X, y, input_dim, extract_features=False):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test)), batch_size=256)

    model.train()
    for _ in range(15):
        for xb, yb in train_loader:
            pred, _ = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    preds, targets, sem_feats = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p, f = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())
            if extract_features:
                sem_feats.extend(f.cpu().numpy())
    acc = accuracy_score(targets, preds)
    return model, acc, np.array(sem_feats) if extract_features else None

# === Train Agents ===
sem_model = CNNSemanticAgent(X_sem.shape[1], dropout=0.3).to(device)
sem_model, acc_sem, sem_features = train_binary_agent(sem_model, X_sem, y_sem, X_sem.shape[1], extract_features=True)
print("Semantic Agent Accuracy:", acc_sem)

beh_model = CNNBehavioralAgent(X_beh.shape[1], dropout=0.3).to(device)
beh_model, acc_beh, beh_features = train_binary_agent(beh_model, X_beh, y_beh, X_beh.shape[1], extract_features=True)
print("Behavioral Agent Accuracy:", acc_beh)

# === Train Decision Agent on matched subset ===
min_len = min(len(sem_features), len(beh_features))
X_dec = np.concatenate([sem_features[:min_len], beh_features[:min_len]], axis=1)
y_dec = y_sem[:min_len]  # Assuming semantic labels are reliable

X_train, X_test, y_train, y_test = train_test_split(X_dec, y_dec, test_size=0.2, stratify=y_dec, random_state=42)
dec_model = DecisionAgent(32, 32, dropout=0.3).to(device)
optimizer = torch.optim.Adam(dec_model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
train_loader = DataLoader(TensorDataset(to_tensor(X_train), to_tensor(y_train)), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(to_tensor(X_test), to_tensor(y_test)), batch_size=256)

# === Train ===
dec_model.train()
for _ in range(15):
    for xb, yb in train_loader:
        pred = dec_model(xb[:, :32], xb[:, 32:])
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

# === Evaluate ===
dec_model.eval()
preds, targets = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        p = dec_model(xb[:, :32], xb[:, 32:])
        preds.extend((p > 0.5).int().cpu().numpy())
        targets.extend(yb.cpu().numpy())

print("Decision Agent Accuracy:", accuracy_score(targets, preds))


Semantic Embedding Shape: (138047, 384)
Semantic Labels Shape: (138047,)
Behavioral Features Shape: (751588, 16)
Behavioral Labels Shape: (751588,)
Semantic Agent Accuracy: 1.0
Behavioral Agent Accuracy: 0.0016764459346186086
Decision Agent Accuracy: 0.0


In [22]:
# === Imports ===
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset

# === Device Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_tensor(x): return torch.tensor(x, dtype=torch.float32, device=device)

# === Load Semantic Dataset ===
df_sem = pd.read_parquet("/home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet")
X_sem = np.stack(df_sem["embedding"].values).astype(np.float32)
y_sem = df_sem["label"].astype(np.float32).values
print("Semantic Embedding Shape:", X_sem.shape)
print("Semantic Labels Shape:", y_sem.shape)

# === Load Behavioral Dataset ===
df_beh = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet")
behavior_columns = [
    'avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance',
    'write_entropy_mean', 'mem_read_entropy_mean', 'mem_write_entropy_mean',
    'mem_readwrite_entropy_mean', 'mem_read_GPA_variance',
    'mem_write_GPA_variance', 'mem_readwrite_GPA_variance',
    'mem_exec_GPA_variance', 'mem_read_count', 'mem_write_count',
    'mem_readwrite_count', 'mem_exec_count'
]
X_beh = df_beh[behavior_columns].to_numpy(dtype=np.float32)
y_beh = df_beh["label"].astype(np.float32).values
print("Behavioral Features Shape:", X_beh.shape)
print("Behavioral Labels Shape:", y_beh.shape)

# === Define Models ===
class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature(x)
        return self.classifier(features).squeeze(), features

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature(x)
        return self.classifier(features).squeeze(), features

class DecisionAgent(nn.Module):
    def __init__(self, sem_dim, beh_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(sem_dim + beh_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, s, b): return self.model(torch.cat([s, b], dim=1)).squeeze()

# === Prepare train/test splits once ===
Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_sem, y_sem, test_size=0.2, stratify=y_sem, random_state=42)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_beh, y_beh, test_size=0.2, stratify=y_beh, random_state=42)

# === Utility Training Function ===
def train_binary_agent(model, X_train, X_test, y_train, y_test, extract_features=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test)), batch_size=256)

    model.train()
    for _ in range(15):
        for xb, yb in train_loader:
            pred, _ = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    preds, targets, feats = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p, f = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())
            if extract_features:
                feats.extend(f.cpu().numpy())
    acc = accuracy_score(targets, preds)
    return model, acc, np.array(feats) if extract_features else None

# === Train Agents ===
sem_model = CNNSemanticAgent(Xs_train.shape[1], dropout=0.3).to(device)
sem_model, acc_sem, sem_features = train_binary_agent(sem_model, Xs_train, Xs_test, ys_train, ys_test, extract_features=True)
print("Semantic Agent Accuracy:", acc_sem)

beh_model = CNNBehavioralAgent(Xb_train.shape[1], dropout=0.3).to(device)
beh_model, acc_beh, beh_features = train_binary_agent(beh_model, Xb_train, Xb_test, yb_train, yb_test, extract_features=True)
print("Behavioral Agent Accuracy:", acc_beh)

# === Train Decision Agent on matched subset ===
min_len = min(len(sem_features), len(beh_features))
X_dec = np.concatenate([sem_features[:min_len], beh_features[:min_len]], axis=1)
y_dec = ys_test[:min_len]  # Use semantic labels from test split

X_train, X_test, y_train, y_test = train_test_split(X_dec, y_dec, test_size=0.2, stratify=y_dec, random_state=42)
dec_model = DecisionAgent(32, 32, dropout=0.3).to(device)
optimizer = torch.optim.Adam(dec_model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
train_loader = DataLoader(TensorDataset(to_tensor(X_train), to_tensor(y_train)), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(to_tensor(X_test), to_tensor(y_test)), batch_size=256)

# === Train Decision Agent ===
dec_model.train()
for _ in range(15):
    for xb, yb in train_loader:
        pred = dec_model(xb[:, :32], xb[:, 32:])
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

# === Evaluate Decision Agent ===
dec_model.eval()
preds, targets = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        p = dec_model(xb[:, :32], xb[:, 32:])
        preds.extend((p > 0.5).int().cpu().numpy())
        targets.extend(yb.cpu().numpy())

print("Decision Agent Accuracy:", accuracy_score(targets, preds))


Semantic Embedding Shape: (138047, 384)
Semantic Labels Shape: (138047,)
Behavioral Features Shape: (751588, 16)
Behavioral Labels Shape: (751588,)
Semantic Agent Accuracy: 1.0
Behavioral Agent Accuracy: 0.9983235540653814
Decision Agent Accuracy: 0.7006519377037306


In [2]:
print("Semantic columns:\n", df_sem.columns)
print("Behavioral columns:\n", df_beh.columns)
print("\nSemantic row count:", len(df_sem))
print("Behavioral row count:", len(df_beh))


Semantic columns:
 Index(['Name', 'Machine', 'Characteristics', 'SizeOfCode',
       'SizeOfInitializedData', 'SizeOfUninitializedData',
       'AddressOfEntryPoint', 'MajorOperatingSystemVersion',
       'MinorOperatingSystemVersion', 'MajorImageVersion', 'MinorImageVersion',
       'MajorSubsystemVersion', 'MinorSubsystemVersion', 'SizeOfImage',
       'SizeOfHeaders', 'Subsystem', 'DllCharacteristics', 'SectionsNb',
       'SectionsMeanEntropy', 'SectionsMinEntropy', 'SectionsMaxEntropy',
       'SectionsMeanRawsize', 'SectionsMinRawsize', 'SectionMaxRawsize',
       'SectionsMeanVirtualsize', 'SectionsMinVirtualsize',
       'SectionMaxVirtualsize', 'ImportsNbDLL', 'ImportsNb',
       'ImportsNbOrdinal', 'ExportNb', 'ResourcesNb', 'ResourcesMeanEntropy',
       'ResourcesMinEntropy', 'ResourcesMaxEntropy', 'ResourcesMeanSize',
       'ResourcesMinSize', 'ResourcesMaxSize', 'LoadConfigurationSize',
       'VersionInformationSize', 'label', 'semantic_text', 'embedding'],
      dtype=

In [ ]:
# === Define Models ===
class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.model(x).squeeze()

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.model(x).squeeze()

class DecisionAgent(nn.Module):
    def __init__(self, sem_dim, beh_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(sem_dim + beh_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, s, b): return self.model(torch.cat([s, b], dim=1)).squeeze()

# === Train Semantic Agent ===
def train_semantic_agent(X, y, input_dim):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    model = CNNSemanticAgent(input_dim, dropout=0.3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train).unsqueeze(1)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test).unsqueeze(1)), batch_size=256)

    model.train()
    for _ in range(15):
        for xb, yb in train_loader:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())
    print("Semantic Agent Accuracy:", accuracy_score(targets, preds))
    return model

# === Train Behavioral Agent ===
def train_behavioral_agent(X, y, input_dim):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    model = CNNBehavioralAgent(input_dim, dropout=0.3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train).unsqueeze(1)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test).unsqueeze(1)), batch_size=256)

    model.train()
    for _ in range(15):
        for xb, yb in train_loader:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())
    print("Behavioral Agent Accuracy:", accuracy_score(targets, preds))
    return model

# === Train Both Agents ===
sem_model = train_semantic_agent(X_sem, y_sem, X_sem.shape[1])
beh_model = train_behavioral_agent(X_beh, y_beh, X_beh.shape[1])



In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from imblearn.over_sampling import SMOTE
import optuna
import random

# === Reproducibility ===
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(42)

# === Device ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_tensor(x): return torch.tensor(x, dtype=torch.float32, device=device)

# === Load Datasets ===
df_sem = pd.read_parquet("/home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet")
X_sem = np.stack(df_sem["embedding"].values).astype(np.float32)
y_sem = df_sem["label"].astype(np.float32).values

df_beh = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet")
behavior_columns = ['avg_read_throughput', 'avg_write_throughput',
    'read_LBA_variance', 'write_LBA_variance',
    'write_entropy_mean', 'mem_read_entropy_mean', 'mem_write_entropy_mean',
    'mem_readwrite_entropy_mean', 'mem_read_GPA_variance',
    'mem_write_GPA_variance', 'mem_readwrite_GPA_variance',
    'mem_exec_GPA_variance', 'mem_read_count', 'mem_write_count',
    'mem_readwrite_count', 'mem_exec_count']  # <-- your 16 columns
X_beh = df_beh[behavior_columns].to_numpy(dtype=np.float32)
y_beh = df_beh["label"].astype(np.float32).values

# === SMOTE on behavioral data ===
sm = SMOTE(random_state=42)
X_beh_bal, y_beh_bal = sm.fit_resample(X_beh, y_beh)

# === Models ===
class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Dropout(dropout), nn.Flatten(),
            nn.Linear((input_dim // 2) * 32, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.model(x).squeeze()

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Dropout(dropout), nn.Flatten(),
            nn.Linear((input_dim // 2) * 16, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.model(x).squeeze()

class DecisionAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.model(x).squeeze()

# === Generic Training Function ===
def train_model(model, X, y, lr, epochs):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    train_loader = DataLoader(TensorDataset(to_tensor(X_train).unsqueeze(1), to_tensor(y_train)), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(to_tensor(X_test).unsqueeze(1), to_tensor(y_test)), batch_size=256)
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            p = model(xb)
            preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())
    return accuracy_score(targets, preds)

# === Optuna Trials ===
results = {}

def objective_sem(trial):
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 10, 30)
    model = CNNSemanticAgent(X_sem.shape[1], dropout).to(device)
    acc = train_model(model, X_sem, y_sem, lr, epochs)
    results['semantic'] = (acc, trial.params)
    return acc

def objective_beh(trial):
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 10, 30)
    model = CNNBehavioralAgent(X_beh_bal.shape[1], dropout).to(device)
    acc = train_model(model, X_beh_bal, y_beh_bal, lr, epochs)
    results['behavioral'] = (acc, trial.params)
    return acc

def objective_dec(trial):
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 10, 30)

    # Fake feature extraction from semantic & behavioral models
    sem_feats = np.random.rand(min(len(X_sem), len(X_beh)), 32)
    beh_feats = np.random.rand(min(len(X_sem), len(X_beh)), 32)
    X_comb = np.concatenate([sem_feats, beh_feats], axis=1)
    y_comb = y_sem[:len(X_comb)]

    model = DecisionAgent(X_comb.shape[1], dropout).to(device)
    acc = train_model(model, X_comb, y_comb, lr, epochs)
    results['decision'] = (acc, trial.params)
    return acc

# === Run Trials ===
print("Optimizing Semantic Agent...")
optuna.create_study(direction="maximize").optimize(objective_sem, n_trials=5)
print("Optimizing Behavioral Agent...")
optuna.create_study(direction="maximize").optimize(objective_beh, n_trials=1000)
print("Optimizing Decision Agent...")
optuna.create_study(direction="maximize").optimize(objective_dec, n_trials=50)

# === Print Results ===
print("\n=== Final Results ===")
for agent, (acc, params) in results.items():
    print(f"\n{agent.capitalize()} Agent:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Best Hyperparameters: {params}")


[I 2025-06-05 12:40:56,101] A new study created in memory with name: no-name-88ed6905-2236-4bf9-95a0-0ba2e0ac6ac1


Optimizing Semantic Agent...


[I 2025-06-05 12:41:31,363] Trial 0 finished with value: 1.0 and parameters: {'dropout': 0.2256657632092981, 'lr': 0.0001673784262552089, 'epochs': 19}. Best is trial 0 with value: 1.0.
[I 2025-06-05 12:42:04,774] Trial 1 finished with value: 1.0 and parameters: {'dropout': 0.4139538369924137, 'lr': 0.00015185523744290247, 'epochs': 20}. Best is trial 0 with value: 1.0.
[I 2025-06-05 12:42:25,908] Trial 2 finished with value: 1.0 and parameters: {'dropout': 0.28030597082181286, 'lr': 0.00022769472722371673, 'epochs': 13}. Best is trial 0 with value: 1.0.
[I 2025-06-05 12:43:12,733] Trial 3 finished with value: 1.0 and parameters: {'dropout': 0.12990784577698308, 'lr': 0.00432059917449103, 'epochs': 29}. Best is trial 0 with value: 1.0.
[I 2025-06-05 12:43:40,024] Trial 4 finished with value: 1.0 and parameters: {'dropout': 0.4366887503843079, 'lr': 0.00023715748585374056, 'epochs': 17}. Best is trial 0 with value: 1.0.
[I 2025-06-05 12:43:40,032] A new study created in memory with name

Optimizing Behavioral Agent...


[I 2025-06-05 12:52:44,792] Trial 0 finished with value: 0.5 and parameters: {'dropout': 0.17043564805027883, 'lr': 0.000299414714688397, 'epochs': 29}. Best is trial 0 with value: 0.5.
[I 2025-06-05 12:58:30,508] Trial 1 finished with value: 0.5520970772859941 and parameters: {'dropout': 0.23159049837391313, 'lr': 0.00511010309640154, 'epochs': 18}. Best is trial 1 with value: 0.5520970772859941.
[I 2025-06-05 13:04:17,528] Trial 2 finished with value: 0.5005031119640692 and parameters: {'dropout': 0.4424274507348257, 'lr': 0.0007510076049700412, 'epochs': 18}. Best is trial 1 with value: 0.5520970772859941.
[W 2025-06-05 13:08:00,009] Trial 3 failed with parameters: {'dropout': 0.31010011037109975, 'lr': 0.006011644403525524, 'epochs': 30} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/henrykabuye/anaconda3/envs/kabuye1_env/lib/python3.9/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(

KeyboardInterrupt: 

In [2]:
# === Imports ===
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from imblearn.over_sampling import SMOTE
import random
import pynvml
import os

# === Set Seed for Reproducibility ===
def set_seed(seed=42):
    """Sets the seed for reproducibility across numpy, torch, and random."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # For multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# === GPU Setup ===
# Removed get_best_gpu_index and related logic that sets torch.cuda.set_device
# DataParallel handles distributing to all visible GPUs starting from cuda:0

use_cuda = torch.cuda.is_available()
num_gpus = torch.cuda.device_count()

# Set the primary device to cuda:0 if CUDA is available, otherwise cpu
# This is the device where the model's parameters will primarily reside
# when DataParallel is used.
device = torch.device("cuda:0" if use_cuda else "cpu")

print(f"Using device: {device}")
print(f"Number of GPUs available: {num_gpus}")

def to_tensor(x, unsqueeze=False):
    """Converts numpy array to a float32 PyTorch tensor."""
    tensor_x = torch.tensor(x, dtype=torch.float32)
    return tensor_x.unsqueeze(1) if unsqueeze else tensor_x

# === Load Semantic Dataset ===
df_sem = None
try:
    df_sem = pd.read_parquet("/home/henrykabuye/Ransap/Datasets/R2RanSAP-20241205T121617Z-001/R2RanSAP/semantic_dataset_with_embeddings.parquet")
    X_sem = np.stack(df_sem["embedding"].values).astype(np.float32)
    y_sem = df_sem["label"].astype(np.float32).values
    print("Semantic Embedding Shape:", X_sem.shape)
    print("Semantic Labels Shape:", y_sem.shape)
except FileNotFoundError:
    print("Error: Semantic dataset not found. Please check the path.")
    exit() # Exit if a critical file is missing
except Exception as e:
    print(f"An error occurred loading semantic dataset: {e}")
    exit()

# === Load Behavioral Dataset ===
df_beh = None
try:
    df_beh = pd.read_parquet("/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet")
    behavior_columns = [
        'avg_read_throughput', 'avg_write_throughput',
        'read_LBA_variance', 'write_LBA_variance',
        'write_entropy_mean', 'mem_read_entropy_mean', 'mem_write_entropy_mean',
        'mem_readwrite_entropy_mean', 'mem_read_GPA_variance',
        'mem_write_GPA_variance', 'mem_readwrite_GPA_variance',
        'mem_exec_GPA_variance', 'mem_read_count', 'mem_write_count',
        'mem_readwrite_count', 'mem_exec_count'
    ]
    X_beh = df_beh[behavior_columns].to_numpy(dtype=np.float32)
    y_beh = df_beh["label"].astype(np.float32).values
    print("Behavioral Features Shape:", X_beh.shape)
    print("Behavioral Labels Shape:", y_beh.shape)
except FileNotFoundError:
    print("Error: Behavioral dataset not found. Please check the path.")
    exit()
except Exception as e:
    print(f"An error occurred loading behavioral dataset: {e}")
    exit()

# === Define Models ===
class CNNSemanticAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim + 1) // 2 * 32 if input_dim % 2 else (input_dim // 2) * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x).squeeze(-1)

class CNNBehavioralAgent(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear((input_dim + 1) // 2 * 16 if input_dim % 2 else (input_dim // 2) * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x).squeeze(-1)

class DecisionAgent(nn.Module):
    def __init__(self, input_dim=1, dropout=0.3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x).squeeze(-1)

# === Train and Evaluate Function ===
def train_and_evaluate_model(model, X, y, epochs=10, lr=1e-3, is_cnn=True):
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

    sm = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = sm.fit_resample(X_train, y_train)

    X_train_tensor = to_tensor(X_train_resampled, unsqueeze=is_cnn)
    y_train_tensor = to_tensor(y_train_resampled)
    X_test_tensor = to_tensor(X_test, unsqueeze=is_cnn)
    y_test_tensor = to_tensor(y_test)

 
    if use_cuda and num_gpus > 1:
        print(f"Wrapping model {model.__class__.__name__} with DataParallel.")
        model = nn.DataParallel(model)
 
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32)

    print(f"Training {model.__class__.__name__}...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device) # Move data to device for each batch
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        # print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

    model.eval()
    preds, targets = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            p = model(xb)
            hard = (p > 0.5).int()
            preds.extend(hard.cpu().numpy())
            # preds.extend((p > 0.5).int().cpu().numpy())
            targets.extend(yb.cpu().numpy())

    acc = accuracy_score(targets, preds)
    prec  = precision_score(targets, preds)
    rec   = recall_score(targets, preds)
    f1    = f1_score(targets, preds)

    print(f"→ Acc: {acc:.4f}   Prec: {prec:.4f}   Rec: {rec:.4f}   F1: {f1:.4f}")
    print("\nFull classification report:\n", classification_report(targets, preds))
    
    return model, acc



# === Train Semantic and Behavioral Models ===
print("\n--- Training Semantic Agent ---")
# Model instantiated here (on CPU by default)
sem_model_instance = CNNSemanticAgent(X_sem.shape[1], dropout=0.3)
sem_model, acc_sem = train_and_evaluate_model(sem_model_instance, X_sem, y_sem, epochs=10, lr=1e-3, is_cnn=True)
print(f"Semantic Agent Accuracy: {acc_sem:.4f}")

print("\n--- Training Behavioral Agent ---")
beh_model_instance = CNNBehavioralAgent(X_beh.shape[1], dropout=0.3)
beh_model, acc_beh = train_and_evaluate_model(beh_model_instance, X_beh, y_beh, epochs=10, lr=1e-3, is_cnn=True)
print(f"Behavioral Agent Accuracy: {acc_beh:.4f}")

# === Train Decision Agent ===
print("\n--- Training Decision Agent ---")
sem_model_eval = sem_model.module if isinstance(sem_model, nn.DataParallel) else sem_model
beh_model_eval = beh_model.module if isinstance(beh_model, nn.DataParallel) else beh_model

sem_model_eval.eval()
beh_model_eval.eval()

with torch.no_grad():
    sem_preds_list = []
    sem_inference_loader = DataLoader(TensorDataset(to_tensor(X_sem, unsqueeze=True)), batch_size=512)
    for xb in sem_inference_loader:
        xb = xb[0].to(device) # Ensure input data for inference is on the correct device
        sem_preds_list.append(sem_model_eval(xb).cpu().numpy())
    sem_pred = np.concatenate(sem_preds_list).reshape(-1, 1)

    beh_preds_list = []
    beh_inference_loader = DataLoader(TensorDataset(to_tensor(X_beh, unsqueeze=True)), batch_size=512)
    for xb in beh_inference_loader:
        xb = xb[0].to(device) # Ensure input data for inference is on the correct device
        beh_preds_list.append(beh_model_eval(xb).cpu().numpy())
    beh_pred = np.concatenate(beh_preds_list).reshape(-1, 1)

# 1. Vertically stack predictions to create a single input feature set
#    This treats each semantic prediction as an independent sample, and
#    each behavioral prediction as an independent sample, for the DecisionAgent.
X_comb = np.vstack([sem_pred, beh_pred])

# 2. Vertically stack labels to match the combined predictions
#    Ensure labels are also column vectors before stacking.
y_comb = np.vstack([y_sem.reshape(-1, 1), y_beh.reshape(-1, 1)])

# 3. Adjust the DecisionAgent's input_dim
#    Since X_comb now has shape (total_samples, 1), the input dimension is 1.
dec_model_instance = DecisionAgent(input_dim=1, dropout=0.3) # <<< input_dim changed to 1

# Now, train the Decision Agent with the combined (vertically stacked) data
dec_model, acc_dec = train_and_evaluate_model(dec_model_instance, X_comb, y_comb, epochs=10, lr=1e-3, is_cnn=False)
print(f"Decision Agent Accuracy: {acc_dec:.4f}")

Using device: cuda:0
Number of GPUs available: 3
Semantic Embedding Shape: (138047, 384)
Semantic Labels Shape: (138047,)
Behavioral Features Shape: (751588, 16)
Behavioral Labels Shape: (751588,)

--- Training Semantic Agent ---
Wrapping model CNNSemanticAgent with DataParallel.
Training DataParallel...


ValueError: too many values to unpack (expected 2)